In [1]:
from modeling_distillemb import BertModel, BertForSequenceClassification, BertForEmbeddingLM
from distill_emb import DistillEmbSmall, DistillEmb
from config import DistillModelConfig, DistillEmbConfig
import torch
from transformers import AutoTokenizer, RwkvConfig, RwkvModel, AutoModel
from tokenizer import CharTokenizer
from knn_classifier import KNNTextClassifier
from data_loader import load_sentiment, load_ner_dataset, load_pos_dataset
from data_loader import load_news_dataset
import pandas as pd
from retrieval import build_json_pairs, top1_accuracy
import os
from transformers import GPT2LMHeadModel

In [2]:
num_input_chars=12

In [3]:
tokenizer = CharTokenizer.from_pretrained(pretrained_directory="logs/distil-emb-base")
distill_config = DistillEmbConfig.from_pretrained(pretrained_model_name_or_path="logs/distil-emb-base")
distill_model = DistillEmb.from_pretrained(pretrained_model_name_or_path="logs/distil-emb-base")

In [4]:
distill_config

DistillEmbConfig {
  "activation": "gelu",
  "architectures": [
    "DistillEmb"
  ],
  "char_vocab_size": 1518,
  "distill_dropout": 0.0,
  "dtype": "float32",
  "embedding_size": 512,
  "model_type": "distilemb",
  "num_input_chars": 12,
  "pad_char_id": 0,
  "size": "base",
  "transformers_version": "4.57.1",
  "use_normalize": false,
  "use_tanh": false
}

In [5]:
# distill_config.distill_dropout = 0.25
config = DistillModelConfig(
    vocab_size=30522,
    hidden_size=512,
    num_hidden_layers=1,
    num_attention_heads=8,
    intermediate_size=3072,
    max_position_embeddings=1024,
    type_vocab_size=2,
    pad_token_id=0,
    position_embedding_type="absolute",
    use_cache=True,
    classifier_dropout=None,
    hidden_dropout_prob=0.1,
    embedding_type="distill",  # 'distilemb', 'fasttext'
    encoder_type='lstm', #'lstm'
    num_input_chars=num_input_chars,  # number of characters in each token
    char_vocab_size=tokenizer.char_vocab_size,
    distill_config=distill_config,
    distill_pretrained_model_name="logs/distil-emb-base",
    is_decoder=False
)


In [6]:
path = "downstream-data/sentiment.parquet"
df = pd.read_parquet(path)
if 'sent' in path:
    # remove 0th index
    df = df[df['text'] != 'tweet'].reset_index(drop=True)

In [7]:
df

,text,label,lang,split
0,Tesfaye ለካስ ጭብል ለብሰሽ የፕሮፌሰርን ፎቶ ለጥፈክ እልም ያልክ ባ...,negative,am,train
1,ይሄው ነው አይደል የእውቀትሽ ጥግ....በሰሚ ሰሚ ከምትናገሪ ለምን ታሪክ...,negative,am,train
2,ዘገበ ይባላል? ሌላ የሚባል ነገር ካለ አንተዉ ንገረን!,negative,am,train
3,?? ድሮ በዘመነ ኮዳክ ፎቶ ቤት ፍላሹ ፏ ሲል አይናችን ተጨፍኖ እንዳይወ...,negative,am,train
4,ዠልጥ?? ???? ገገማ,negative,am,train
...,...,...,...,...
105856,@user Taakkee Jabaadhu!!! olola gadi galoo hin...,positive,or,test
105857,@user Waraana Bilisummaa Oromiyaa. Unity of Or...,neutral,or,test
105858,#Jawwaar dhugumatti hogganaa walitti-hidhaa ga...,negative,or,test
105859,Yooyyaa Yooyyaa akkam jirtan sabni Oromo hundi...,negative,or,test


In [8]:
len(df['lang'].unique())

14

In [9]:
lang_counts = df.groupby('split')['lang'].nunique()
for split, count in lang_counts.items():
    print(f"{split.capitalize()} split has {count} languages.")

Dev split has 12 languages.
Test split has 14 languages.
Train split has 12 languages.


In [10]:
label2id = {label: idx for idx, label in enumerate(sorted(df['label'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}

df['label'] = df['label'].map(label2id).astype(int)

config.label2id = label2id
config.id2label = id2label

print(f"Converted labels to integers: {label2id}")

Converted labels to integers: {'negative': 0, 'neutral': 1, 'positive': 2}


In [11]:
num_labels = len(df['label'].unique())
config.num_labels = num_labels
model = BertForSequenceClassification(config)

In [12]:
labels = [x.item() for x in df['label'].unique()]
print(labels)
text_col = 'text'

[0, 1, 2]


In [13]:
from datasets import Dataset, DatasetDict
df['text'] = df[text_col]

def build_augmented_dataset(dataframe: pd.DataFrame, samples_per_row: int = 1, separator: str = " "):
    sentiment_aliases = {
        "negative": ("negative", "neg", "0"),
        "neutral": ("neutral", "neu", "1"),
        "positive": ("positive", "pos", "2"),
    }

    def canonical_name(label_id: int) -> str:
        label_name = id2label[label_id].lower()
        for canonical, aliases in sentiment_aliases.items():
            if any(alias in label_name for alias in aliases):
                return canonical
        return label_name

    canonical_to_id = {canonical_name(lbl): lbl for lbl in dataframe["label"].unique()}

    def resolve_label(label_a: int, label_b: int) -> int:
        if 0 in (label_a, label_b):
            return 0
        if 1 in (label_a, label_b):
            return 1
        return 2

    augmented_rows = []
    for _, row in dataframe.iterrows():
        base_text, base_label = row["text"], row["label"]
        augmented_rows.append({"text": base_text, "label": base_label})

        if samples_per_row < 1:
            continue
        
        sampled = dataframe.sample(n=10, replace=True)
        for _, sampled_row in sampled.iterrows():
            combined_text = f"{base_text}{separator}{sampled_row['text']}"
            combined_label = resolve_label(base_label, sampled_row["label"])
            augmented_rows.append({"text": combined_text, "label": combined_label})

    augmented_df = pd.DataFrame(augmented_rows)
    # remove duplicates on text
    augmented_df = augmented_df.drop_duplicates(subset=['text']).reset_index(drop=True)
    min_count = augmented_df["label"].value_counts().min()
    augmented_df = (
        augmented_df.groupby("label", group_keys=False)
        .apply(lambda group: group.sample(n=min_count, random_state=42))
        .reset_index(drop=True)
    )
    return augmented_df
# Assuming df is your dataframe
# Split the data based on the 'split' column
train_df = df[df['split'] == 'train'][['text', 'label']]
test_df = df[df['split'] == 'test'][['text', 'label']]

aug_train_df = build_augmented_dataset(train_df, samples_per_row=2, separator=" ")

/tmp/ipykernel_4023996/1436789007.py:47: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.sample(n=min_count, random_state=42))


In [14]:
aug_train_df

,text,label
0,Igi ọ̀mọ̀ ni àwọn àgbà fi ń gbẹ́ ère òrìṣà. #E...,0
1,i don tire to dey look for the x wey nor dey e...,0
2,@user @user @user @user @user @user @user @use...,0
3,مملكة الدعارة خخخخخخخخخخخخخخخخخخ sister efya w...,0
4,وعلاش ياليل بليتيني بالتخمام ! @user Chukwu go...,0
...,...,...
257398,الواجهة البحرية بومارشي - جيجل 💚\n#elyaatouris...,2
257399,@user Innalillahi 🤔wllh kuji tsoron Allah bros...,2
257400,😂😂😂😂 Olohun ma se wa ni irin ise ESU #TweetinY...,2
257401,ኢህአዴግ የሚለው ስም መጥፋቱ ብቻ ለብልፅግና ክብር እንድሰጥ አድርጎኛል ...,2


In [ ]:
train_df

In [ ]:
# Create HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset

In [ ]:
from typing import Dict, Any

def preprocess_function(examples: Dict[str, Any]):
    batch = tokenizer(
        examples["text"],
        padding=False,
        max_length=256,
        return_attention_mask=False,
    )

    batch["labels"] = examples["label"]
    return batch



tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

In [ ]:
len(train_dataset[0]['text'].split())

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
class CustomDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch = self.tokenizer.pad(
            features,
            padding="longest",
            max_length=256,
            return_tensors="pt",
            return_attention_mask=True,
            padding_side="right"
        )
        return batch

data_collator = CustomDataCollator(tokenizer)

In [ ]:
##### from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted', labels=labels)
    f1_macro = f1_score(labels, predictions, average='macro', labels=labels)
    f1_micro = f1_score(labels, predictions, average='micro', labels=labels)
    return {"accuracy": acc, "f1_weighted": f1, "f1_macro": f1_macro, "f1_micro": f1_micro}


import os
dataloader_num_workers=os.cpu_count() - 1
batch_size = 32

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=15,
    weight_decay=0.01,
    report_to=[],
    eval_strategy="epoch",  
    save_total_limit=1,
    save_only_model=True,
    logging_strategy="steps",
    logging_steps=10,
    label_smoothing_factor=0.1,
    max_grad_norm=5.0,
    warmup_ratio=0.0,
    lr_scheduler_type="cosine",
    dataloader_num_workers=16,        # Number of CPU workers for data loading
    dataloader_pin_memory=True,      # Faster GPU transfer
    gradient_accumulation_steps=4
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Evaluate the model after training
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

In [ ]:
trainer.evaluate()

In [ ]:
# model = BertForSequenceClassification.from_pretrained("distil-emb-seqcls-lstm").cuda()
model = trainer.model
model.eval()
# Ensure 'language' column exists in df
test_df = df[df['split'] == 'test'][['text', 'label', 'lang']]
languages = test_df['lang'].unique()
per_language_f1 = {}

batch_size = 16

for lang in languages:
    lang_df = test_df[test_df['lang'] == lang]
    texts = lang_df['text'].tolist()
    labels = lang_df['label'].values
    preds = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]
        tokenized = tokenizer(
            batch_texts,
            padding='longest',
            truncation=True,
            max_length=256,
            return_tensors="pt",
            return_attention_mask=True,
            padding_side="right"
        )
        with torch.no_grad():
            inputs = {k: v.cuda() for k, v in tokenized.items()}
            outputs = model(**inputs)
            batch_preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            preds.extend(batch_preds)
    f1 = f1_score(labels, preds, average='weighted', labels=labels)
    per_language_f1[lang] = f1

# Print per-language F1
for lang, f1 in per_language_f1.items():
    print(f"Language: {lang}, F1: {f1:.4f}")

# Average F1
average_f1 = sum(per_language_f1.values()) / len(per_language_f1)
print(f"Average F1 across languages: {average_f1:.4f}")

In [ ]:
model.save_pretrained("distil-emb-news-lstm-best-256")